In [ ]:
import torch
import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader
from facenet_pytorch import InceptionResnetV1
import os
import pickle
import numpy as np

In [ ]:
# Muat embeddings dari folder train yang sudah dibuat sebelumnya
try:
    with open('embeddings_augmentation.pkl', 'rb') as f:
        previous_embeddings = pickle.load(f)
    print("Embeddings lama berhasil dimuat.")
except FileNotFoundError:
    print("File embeddings_augment.pkl tidak ditemukan.")
    exit()
except Exception as e:
    print(f"Error saat memuat embeddings.pkl: {e}")
    exit()

Embeddings lama berhasil dimuat.


In [49]:
# Transformasi gambar
transform = transforms.Compose([
    transforms.Resize((256, 256)),  # Ukuran input yang dibutuhkan oleh model
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])  # Normalisasi sesuai dengan requirement FaceNet
])

In [ ]:
# buat variabel untuk menampung Folder data
data_dir = 'sample_train_augmented'

In [ ]:
# Membuat dataset custom dan dataloader (batch data)
# ImageFolder menyimpan label dalam bentuk int 
dataset = datasets.ImageFolder(data_dir, transform=transform)
dataloader = DataLoader(dataset, batch_size=32, shuffle=False)

In [52]:
# Memastikan dataset memiliki data
if len(dataset) == 0:
    print("Dataset kosong. Pastikan folder sample_train_augment berisi gambar dengan format yang benar.")
    exit()

print(f"Dataset memiliki {len(dataset)} gambar dari {len(dataset.classes)} kelas.")

Dataset memiliki 2292 gambar dari 191 kelas.


In [ ]:
# Imagefolder juga punya atribut untuk menyimpan label dengan string label
class_names = dataset.classes

# check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Inisialisasi model
model = InceptionResnetV1(pretrained='vggface2').eval().to(device)

In [ ]:
# Menghitung Embeddings baru menggunakan data augmentasi
new_embeddings = {}
with torch.no_grad():
    for imgs, labels in dataloader:
        # pindahkan imgs ke device yang ada 
        imgs = imgs.to(device)
        # hitung embedding 
        outputs = model(imgs)

        # masukan embedding ke dictionary 
        # lakukan looping label int 
        for i, label in enumerate(labels):
            # ambil nama label asli melalui label int 
            label_name = class_names[label]
            # cek apakah label ada di dictionary 
            if label_name not in new_embeddings:
                new_embeddings[label_name] = []
            new_embeddings[label_name].append(outputs[i].cpu().numpy())

In [56]:
# Gabungkan embeddings lama dan baru
for label_name, embed_list in new_embeddings.items():
    if label_name in previous_embeddings:
        previous_embeddings[label_name].extend(embed_list)
        print(f"Menambahkan {len(embed_list)} embedding ke label '{label_name}'. Total sekarang: {len(previous_embeddings[label_name])}")
    else:
        previous_embeddings[label_name] = embed_list
        print(f"Label baru '{label_name}' ditambahkan dengan {len(embed_list)} embedding.")

Menambahkan 12 embedding ke label 'S001'. Total sekarang: 28
Menambahkan 12 embedding ke label 'S006'. Total sekarang: 16
Menambahkan 12 embedding ke label 'S008'. Total sekarang: 24
Menambahkan 12 embedding ke label 'S013'. Total sekarang: 14
Menambahkan 12 embedding ke label 'S015'. Total sekarang: 16
Menambahkan 12 embedding ke label 'S022'. Total sekarang: 14
Menambahkan 12 embedding ke label 'S023'. Total sekarang: 16
Menambahkan 12 embedding ke label 'S027'. Total sekarang: 14
Menambahkan 12 embedding ke label 'S029'. Total sekarang: 14
Menambahkan 12 embedding ke label 'S030'. Total sekarang: 18
Menambahkan 12 embedding ke label 'S031'. Total sekarang: 20
Menambahkan 12 embedding ke label 'S032'. Total sekarang: 16
Menambahkan 12 embedding ke label 'S033'. Total sekarang: 24
Menambahkan 12 embedding ke label 'S036'. Total sekarang: 16
Menambahkan 12 embedding ke label 'S044'. Total sekarang: 26
Menambahkan 12 embedding ke label 'S045'. Total sekarang: 14
Menambahkan 12 embedding

In [ ]:
# Menyimpan embeddings gabungan ke file pickle baru
with open('embeddings_incremental.pkl', 'wb') as f:
    pickle.dump(previous_embeddings, f)
print("Embeddings gabungan berhasil disimpan dalam embeddings_incremental.pkl")

Embeddings gabungan berhasil disimpan dalam combined_embeddings7.pkl
